# Multiple ROI Selector

Interactive Allen CCF ROI viewer for Research Associates.

This interface supports:

- Allen CCF regions
- composite regions from `config/custom_regions.json`
- curated drawn masks from `config/custom_masks.json`
- multiple ROIs
- multiple anatomical regions per ROI
- one display color per ROI
- reusable saved ROI plans
- Allen Atlas plate navigation

The brain viewer is displayed only after clicking **Load selected regions**.

In [ ]:
## 1. Development helpers
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import html

from pathlib import Path
from string import ascii_uppercase

import ipywidgets as widgets
from IPython.display import display, clear_output

from ccf_roi_selector.atlas import load_atlas
from ccf_roi_selector.plotting import show_overlay_slice
from ccf_roi_selector.roi import (
    load_custom_regions,
    load_custom_masks_registry,
)

In [ ]:
## 3. Load Allen CCF atlas
template, annotation, parcellation_annotation = load_atlas()

# template.shape, annotation.shape, len(parcellation_annotation)

In [ ]:
# ## 4. Load available regions

# The selector combines three sources:

# 1. Allen CCF regions
# 2. composite regions from `custom_regions.json`
# 3. manually curated masks from `custom_masks.json`

custom_regions = load_custom_regions()
custom_masks = load_custom_masks_registry()

allen_regions = (
    parcellation_annotation[
        "parcellation_term_acronym"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

available_regions = sorted(
    set(allen_regions)
    | set(custom_regions.keys())
    | set(custom_masks.keys())
)

# print(f"Allen regions: {len(allen_regions)}")
# print(f"Composite custom regions: {len(custom_regions)}")
# print(f"Drawn custom masks: {len(custom_masks)}")
# print(f"Total selectable regions: {len(available_regions)}")

# if custom_masks:
#     print("Drawn custom regions:", sorted(custom_masks.keys()))

In [ ]:
## 5. Repository paths
cwd = Path.cwd()

if cwd.name == "notebooks":
    repo_root = cwd.parent
else:
    repo_root = cwd

plans_dir = repo_root / "plans"
plans_dir.mkdir(parents=True, exist_ok=True)

# print("Plans folder:", plans_dir)

In [ ]:
## 6. Interface state
roi_editors = []

current_roi_plan = []
current_roi_colors = {}

roi_container = widgets.VBox()

selection_status = widgets.HTML()
plan_status = widgets.HTML()

viewer_container = widgets.VBox()

In [ ]:
## 7. Selected-region display for each ROI
def refresh_selected_regions(editor):

    region_widgets = []

    for region in editor["regions"]:

        region_label = widgets.Label(
            value=region,
            layout=widgets.Layout(width="210px")
        )

        remove_button = widgets.Button(
            description="×",
            tooltip=f"Remove {region}",
            layout=widgets.Layout(
                width="35px",
                height="30px"
            )
        )

        def remove_region(
            button,
            region=region,
            editor=editor
        ):
            if region in editor["regions"]:
                editor["regions"].remove(region)

            refresh_selected_regions(editor)

        remove_button.on_click(remove_region)

        region_widgets.append(
            widgets.HBox(
                [
                    region_label,
                    remove_button
                ]
            )
        )

    if region_widgets:

        editor["regions_box"].children = region_widgets

    else:

        editor["regions_box"].children = [
            widgets.HTML(
                "<i>No regions selected yet.</i>"
            )
        ]

In [ ]:
## 8. Refresh ROI cards
def refresh_roi_container():

    roi_widgets = []

    for i, editor in enumerate(roi_editors):

        if i < len(ascii_uppercase):
            label = ascii_uppercase[i]
        else:
            label = str(i + 1)

        editor["title"].value = (
            f"<h3>ROI {label}</h3>"
        )

        roi_widgets.append(
            editor["widget"]
        )

    roi_container.children = roi_widgets

In [ ]:
# ## 9. Create an ROI editor

# Each ROI can contain one or more regions and has one color.

def add_roi(button=None, initial_data=None):

    if initial_data is None:
        initial_data = {}

    title = widgets.HTML()

    name_input = widgets.Text(
        value=initial_data.get("name", ""),
        placeholder="Enter ROI name...",
        description="Name:",
        layout=widgets.Layout(width="500px")
    )

    region_input = widgets.Combobox(
        options=available_regions,
        placeholder="Search region...",
        description="Region:",
        ensure_option=False,
        layout=widgets.Layout(width="500px")
    )

    add_region_button = widgets.Button(
        description="Add Region",
        button_style="info",
        layout=widgets.Layout(width="180px")
    )

    regions_box = widgets.VBox()

    color_picker = widgets.ColorPicker(
        concise=False,
        description="Color:",
        value=initial_data.get("color", "#ff0000"),
        layout=widgets.Layout(width="430px")
    )

    remove_roi_button = widgets.Button(
        description="Remove ROI",
        icon="trash",
        button_style="danger",
        layout=widgets.Layout(width="150px")
    )

    editor = {
        "title": title,
        "name": name_input,
        "region_input": region_input,
        "regions": list(
            initial_data.get("regions", [])
        ),
        "regions_box": regions_box,
        "color": color_picker,
        "remove_button": remove_roi_button,
    }

    def add_region_clicked(button):

        region = region_input.value.strip()

        if not region:
            return

        if region not in available_regions:

            selection_status.value = (
                "<span style='color:red;'>"
                f"Region '{html.escape(region)}' "
                "was not found."
                "</span>"
            )

            return

        if region not in editor["regions"]:
            editor["regions"].append(region)

        region_input.value = ""

        selection_status.value = ""

        refresh_selected_regions(editor)

    add_region_button.on_click(
        add_region_clicked
    )

    def remove_roi_clicked(button):

        if editor in roi_editors:
            roi_editors.remove(editor)

        refresh_roi_container()

    remove_roi_button.on_click(
        remove_roi_clicked
    )

    editor["widget"] = widgets.VBox(
        [
            title,

            name_input,

            widgets.HBox(
                [
                    region_input,
                    add_region_button
                ]
            ),

            widgets.HTML(
                "<b>Selected regions:</b>"
            ),

            regions_box,

            color_picker,

            remove_roi_button,

            widgets.HTML("<hr>")
        ],
        layout=widgets.Layout(
            width="780px",
            padding="10px"
        )
    )

    roi_editors.append(editor)

    refresh_selected_regions(editor)
    refresh_roi_container()

In [ ]:
## 10. Add ROI button
add_roi_button = widgets.Button(
    description="+ Add ROI",
    button_style="success",
    layout=widgets.Layout(width="160px")
)

add_roi_button.on_click(add_roi)

In [ ]:
# ## 11. Collect the current ROI plan

# The interface representation is converted to a simple JSON-compatible structure.
def collect_roi_plan():

    plan = []

    for i, editor in enumerate(roi_editors):

        if not editor["regions"]:
            continue

        name = editor["name"].value.strip()

        if not name:

            if i < len(ascii_uppercase):
                name = f"ROI {ascii_uppercase[i]}"
            else:
                name = f"ROI {i + 1}"

        roi = {
            "name": name,
            "regions": list(
                editor["regions"]
            ),
            "color": editor["color"].value
        }

        plan.append(roi)

    return plan

In [ ]:
# ## 12. Convert ROI plan to plotting colors

# `plotting.py` receives a `region -> color` mapping.

# If multiple regions belong to the same ROI, each receives the ROI's color.
def plan_to_roi_colors(plan):

    roi_colors = {}

    for roi in plan:

        color = roi["color"]

        for region in roi["regions"]:
            roi_colors[region] = color

    return roi_colors

In [ ]:
## 13. Save ROI plans
plan_name_input = widgets.Text(
    placeholder="e.g. thalamus_regions",
    description="Plan name:",
    layout=widgets.Layout(width="420px")
)

save_plan_button = widgets.Button(
    description="Save Plan",
    icon="save",
    button_style="success",
    layout=widgets.Layout(width="140px")
)

In [ ]:
def refresh_plan_dropdown():

    files = sorted(
        p.name
        for p in plans_dir.glob("*.json")
    )

    plan_dropdown.options = files

In [ ]:
def save_plan_clicked(button):

    name = plan_name_input.value.strip()

    if not name:

        plan_status.value = (
            "<span style='color:red;'>"
            "Enter a plan name first."
            "</span>"
        )

        return

    plan = collect_roi_plan()

    if not plan:

        plan_status.value = (
            "<span style='color:red;'>"
            "Add at least one region before saving."
            "</span>"
        )

        return

    plan_path = (
        plans_dir
        / f"{name}.json"
    )

    data = {
        "version": 1,
        "rois": plan
    }

    with open(
        plan_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            data,
            f,
            indent=4
        )

    plan_status.value = (
        "<span style='color:green;'>"
        f"✓ Saved {html.escape(plan_path.name)}"
        "</span>"
    )

    refresh_plan_dropdown()


save_plan_button.on_click(
    save_plan_clicked
)

In [ ]:
## 14. Load saved ROI plans
plan_dropdown = widgets.Dropdown(
    description="Saved plan:",
    options=[],
    layout=widgets.Layout(width="420px")
)

load_plan_button = widgets.Button(
    description="Load Plan",
    icon="folder-open",
    layout=widgets.Layout(width="140px")
)

In [ ]:
def load_plan_clicked(button):

    filename = plan_dropdown.value

    if not filename:

        plan_status.value = (
            "<span style='color:red;'>"
            "No saved plan selected."
            "</span>"
        )

        return

    plan_path = (
        plans_dir
        / filename
    )

    with open(
        plan_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    # Preferred current format
    if "rois" in data:

        rois = data["rois"]

    # Backward compatibility with early roi_colors plans
    elif "roi_colors" in data:

        rois = []

        for region, color in data["roi_colors"].items():

            rois.append(
                {
                    "name": region,
                    "regions": [region],
                    "color": color
                }
            )

    else:

        plan_status.value = (
            "<span style='color:red;'>"
            "This JSON file is not a recognized ROI plan."
            "</span>"
        )

        return

    roi_editors.clear()

    refresh_roi_container()

    for roi in rois:
        add_roi(
            initial_data=roi
        )

    plan_status.value = (
        "<span style='color:green;'>"
        f"✓ Loaded {html.escape(filename)}"
        "</span>"
    )


load_plan_button.on_click(
    load_plan_clicked
)

refresh_plan_dropdown()

In [ ]:
## 15. Allen Atlas plate helpers
def max_plate_from_template(template):

    total_slices = template.shape[2]

    return (
        (total_slices - 1) // 4
    ) + 1


def plate_to_slice(plate):

    slice_index = (
        (plate - 1) * 4
    )

    return min(
        slice_index,
        template.shape[2] - 1
    )

In [ ]:
## 16. Brain viewer controls
max_plate = max_plate_from_template(
    template
)

default_plate = min(
    53,
    max_plate
)

plate_slider = widgets.IntSlider(
    min=1,
    max=max_plate,
    step=1,
    value=default_plate,
    description="Atlas Plate:",
    continuous_update=True,
    layout=widgets.Layout(width="700px")
)

prev_button = widgets.Button(
    description="← Previous Plate"
)

next_button = widgets.Button(
    description="Next Plate →"
)

prev_button.style.button_color = "#d9eaf7"
next_button.style.button_color = "#d9eaf7"

prev_button.layout.margin = "0 5px"
next_button.layout.margin = "0 5px"

brain_output = widgets.Output()

brain_summary = widgets.HTML()

In [ ]:
## 17. Render the currently selected atlas plate
def render_current_plate(change=None):

    if not current_roi_colors:
        return

    plate = plate_slider.value

    slice_index = plate_to_slice(
        plate
    )

    with brain_output:

        clear_output(wait=True)

        show_overlay_slice(
            template,
            annotation,
            parcellation_annotation,
            current_roi_colors,
            slice_index=slice_index,
            title=f"Allen Atlas Plate: {plate}"
        )

In [ ]:
def previous_plate(button):

    if plate_slider.value > plate_slider.min:
        plate_slider.value -= 1


def next_plate(button):

    if plate_slider.value < plate_slider.max:
        plate_slider.value += 1


prev_button.on_click(
    previous_plate
)

next_button.on_click(
    next_plate
)

plate_slider.observe(
    render_current_plate,
    names="value"
)

In [ ]:
# ## 18. Load selected regions into the brain viewer

# The viewer remains hidden until this button is pressed.
load_selected_button = widgets.Button(
    description="Load selected regions",
    icon="check",
    button_style="primary",
    layout=widgets.Layout(
        width="250px",
        height="45px"
    )
)

In [ ]:
def load_selected_regions(button):

    global current_roi_plan
    global current_roi_colors

    plan = collect_roi_plan()

    if not plan:

        selection_status.value = (
            "<span style='color:red;'>"
            "Please select at least one region first."
            "</span>"
        )

        viewer_container.children = []

        return

    current_roi_plan = plan

    current_roi_colors = (
        plan_to_roi_colors(plan)
    )

    selection_status.value = (
        "<span style='color:green;'>"
        "✓ Selected regions loaded."
        "</span>"
    )

    summary_lines = []

    for roi in plan:

        regions = ", ".join(
            roi["regions"]
        )

        summary_lines.append(
            f'''
            <div style="margin-bottom:4px;">
                <span style="
                    display:inline-block;
                    width:12px;
                    height:12px;
                    background:{roi["color"]};
                    margin-right:6px;
                    border:1px solid #777;
                "></span>

                <b>{html.escape(roi["name"])}</b>:
                {html.escape(regions)}
            </div>
            '''
        )

    brain_summary.value = (
        "<h3>Loaded ROIs</h3>"
        + "".join(summary_lines)
    )

    navigation_buttons = widgets.HBox(
        [
            prev_button,
            next_button
        ],
        layout=widgets.Layout(
            width="700px",
            justify_content="center"
        )
    )

    viewer_container.children = [
        widgets.HTML("<hr>"),
        brain_summary,
        navigation_buttons,
        plate_slider,
        brain_output
    ]

    render_current_plate()


load_selected_button.on_click(
    load_selected_regions
)

In [ ]:
## 19. Build the Research Associate interface

# Start with one empty ROI card
if not roi_editors:
    add_roi()


roi_controls = widgets.VBox(
    [
        widgets.HTML(
            "<h2>ROI Selector</h2>"
            "<p>"
            "Add one or more anatomical regions to each ROI "
            "and choose a display color."
            "</p>"
        ),

        roi_container,

        add_roi_button,

        selection_status,

        widgets.HTML("<hr>"),

        widgets.HTML(
            "<h3>ROI Plans</h3>"
        ),

        widgets.HBox(
            [
                plan_name_input,
                save_plan_button
            ]
        ),

        widgets.HBox(
            [
                plan_dropdown,
                load_plan_button
            ]
        ),

        plan_status,

        widgets.HTML("<hr>"),

        load_selected_button
    ],
    layout=widgets.Layout(
        width="820px"
    )
)


app = widgets.VBox(
    [
        roi_controls,
        viewer_container
    ]
)


display(app)

## Notes

A region such as `TH:Ant-MM` should now appear automatically in the Region search box if it exists in `config/custom_masks.json`.

The user-facing interface does not need to know whether a selected region comes from:

- the Allen CCF annotation
- `custom_regions.json`
- a manually curated `.npz` mask

That distinction is handled by `roi.py` and `plotting.py`.

<hr>

<div style="text-align: center; color: #777; font-size: 13px; margin-top: 25px;">
    Created by <b>Camila Vergara</b> ·
    <a href="https://github.com/macavero" target="_blank">
        GitHub
    </a>
</div>